# CropCheckUp — BiRefNet on padded close-ups

The first BiRefNet run kept 928 images whole (`fallback_full_image`, excluding Corn healthy and Common rust, which
need no segmentation). Most are close-ups where the leaf fills the frame, so BiRefNet finds no distinct subject; the
rest are small leaves whose correct masks fell under the 10% coverage cutoff.

Each image is padded with a border of its own board colour (plus a little noise, so the border isn't a flat patch)
at several ratios, which shrinks the leaf to an object inside the frame, and BiRefNet runs on each. No single ratio
works for every image (a test on 40 showed small leaves need none, most close-ups 1.5-2, and 3 makes BiRefNet pick
the whole photo square), so the mask that best agrees (IoU) with a rough colour mask is kept. The colour mask is too
blocky to use as a cutout but reliably rules out empty, lesion-only and whole-frame masks. Images where no ratio
reaches `MIN_IOU` keep the whole image, as before; that's most of Corn Northern Leaf Blight.

Output: `segmented_padded.zip` (`segmented/<class>/<name>.png`, same RGBA format as the first run) and
`manifest_padded.json` (chosen ratio and IoU per image, and the images kept whole).

In [ ]:
import subprocess, sys

# BiRefNet's remote code doesn't load under transformers 5.x.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40,<5", "timm", "kornia", "einops"])

In [ ]:
import json
from pathlib import Path

RATIOS = [1.0, 1.25, 1.5, 1.75, 2.0]  # 1.0 is unpadded, for the small leaves
MIN_IOU = 0.75  # in the 40-image test, good masks scored >= 0.79 and bad ones <= 0.72
MODEL_ID = "ZhengPeng7/BiRefNet"
IMG_SIZE = 1024
BATCH_SIZE = 8
DEFAULT_BOARD = (142, 136, 144)  # median board colour over these images, for frames with too little board to measure

RAW_DIR = next(p for p in Path("/kaggle/input").rglob("raw") if p.parent.name == "plantvillage")
WORK_DIR = Path("/kaggle/working")

# class -> image stems, from the local manifest's fallback_full_image after remove_bad_images.py
IMAGES = {
 "Apple___Apple_scab": [
  "01f3deaa-6143-4b6c-9c22-620a46d8be04___FREC_Scab 3112",
  "14c623e5-051c-42f6-9e4f-f7a93e6a723c___FREC_Scab 2965",
  "15bfa086-7d4a-43c0-800d-2fe8255ba50c___FREC_Scab 3150",
  "45000608-507a-46f5-be52-3ee24d0fb47d___FREC_Scab 3113",
  "bceb695a-e877-4b2b-bf05-241661d511d4___FREC_Scab 3240"
 ],
 "Blueberry___healthy": [
  "9183f1d7-5b88-492d-9d8f-1a72106894c1___RS_HL 2518",
  "b8c8acd9-5102-479f-ba9c-e7472b204740___RS_HL 2597"
 ],
 "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": [
  "00120a18-ff90-46e4-92fb-2b7a10345bd3___RS_GLSp 9357",
  "0140764c-6157-4995-9ada-9c10b81af3b8___RS_GLSp 4378",
  "0157de0c-5b81-44a1-abe8-eecaa6f1261a___RS_GLSp 4287",
  "01a02916-5d4c-4256-936b-6b1a9ee65b11___RS_GLSp 9276",
  "05f92471-3cd4-441b-af21-1a02304d0b6c___RS_GLSp 7315",
  "0657e6c0-5a6d-4ef3-807a-4e31e57fb683___RS_GLSp 7344 copy",
  "06785548-9756-4410-b3c2-55cc089276ce___RS_GLSp 4288",
  "07e00695-1271-4cf1-b08d-63e27969b03d___RS_GLSp 9312",
  "07f54257-0ec8-4bce-991f-90d46646d4f8___RS_GLSp 4378 copy 2",
  "0a01cc10-3892-4311-9c48-0ac6ab3c7c43___RS_GLSp 9352",
  "0c5973d2-b9e1-473c-a5d6-8e9854b918b9___RS_GLSp 9340",
  "0d13a6d5-7993-4a5d-beb1-4f38b7805fac___RS_GLSp 9303",
  "0e0ed08d-3021-49a7-9098-7c90afeb2fd5___RS_GLSp 4346",
  "0f36d500-e463-44ef-82ae-fb1ca5012335___RS_GLSp 9321",
  "137762d7-fcd5-405c-a4d5-4096321951d0___RS_GLSp 9345",
  "14913b5b-f425-43f4-8266-e6c353f96781___RS_GLSp 4328",
  "165f35a9-5e03-4ec8-9167-bd346c863e41___RS_GLSp 4289 copy 3",
  "1836cc95-03e5-4430-8cc1-cca482b94a0d___RS_GLSp 4289 copy",
  "186c5f5d-a8d7-40bf-abe7-cebefa66808c___RS_GLSp 4355 copy 2",
  "1b1991ac-4e5c-4528-ac3d-096d63fb05aa___RS_GLSp 4491",
  "1b3e0f15-bbfb-4629-840f-310c7bf2a17c___RS_GLSp 9335",
  "2045dd77-e896-4c78-b40b-5b93631e4e5c___RS_GLSp 9300",
  "206517f4-4d5f-4d16-abab-7e634fc78d0d___RS_GLSp 4412",
  "22e876a7-46e4-4b87-b2ac-0668117fb455___RS_GLSp 4469",
  "24098661-633d-46fb-9542-a12a41dd3223___RS_GLSp 4472",
  "27ea3780-cdea-449f-b427-04ad3d0a7eae___RS_GLSp 9271",
  "29d60537-2717-45e1-b608-c19270302f67___RS_GLSp 4558",
  "29f064ba-c8cd-47f7-add1-24fe1263f2db___RS_GLSp 9319",
  "2bfff79a-08ce-4f07-a059-465326ef2ea2___RS_GLSp 4436",
  "2c890074-1b01-46cb-9e56-8601d13ec218___RS_GLSp 4469 copy",
  "302d1d00-fe10-4ec1-878c-5ac721c6ec44___RS_GLSp 9356",
  "30554ff6-3eef-4d17-8304-de6a8a478f2b___RS_GLSp 4683",
  "338c8024-b930-49fa-bf1f-21d7874fb7fb___RS_GLSp 4319",
  "348aafc3-4309-4507-b6b3-cb037711ca48___RS_GLSp 4399",
  "34ed1bf6-87eb-49dd-bc0b-fd9d0bd3dd0c___RS_GLSp 4378 copy",
  "36584bae-f3bb-40c0-b866-ed5ba0c2f7ae___RS_GLSp 4577",
  "398cdfcf-3bc5-486c-ab4e-8bce2f2bbc23___RS_GLSp 9333",
  "3af22715-5f08-4b3b-8c88-6264a73accba___RS_GLSp 7293",
  "3bc17d8a-7efc-469a-a0af-aa8a68e62e89___RS_GLSp 9323",
  "3bc19f2c-2ab4-4a1c-bd0f-e0a322058b8b___RS_GLSp 4423",
  "3c70ede6-fd2c-4e9c-8c2c-2d6a369564d4___RS_GLSp 4345",
  "3e52b140-6513-4a97-8ee8-9935a7d4b8b8___RS_GLSp 9279",
  "3fefc48b-6431-4c5c-acc7-19217866da5b___RS_GLSp 9292",
  "40dceef3-f53b-4132-87c0-69bf1046f8c4___RS_GLSp 9362",
  "40de2256-a1de-4781-830b-d9a06544fe06___RS_GLSp 4398",
  "41759050-8a14-4394-9558-c74ec19a5e50___RS_GLSp 4685",
  "468d93ab-7a44-4ae7-a489-a03d39651009___RS_GLSp 4466",
  "48beb71e-555f-4510-9951-a57f7edce204___RS_GLSp 9334",
  "4be4a686-ab64-43d7-9d7c-0939e389aac9___RS_GLSp 9338",
  "4c7894c0-29a5-4643-85ff-95ddb85dbd81___RS_GLSp 4605",
  "4f50bad3-aee2-4d6d-a144-5c56f4c1fdd9___RS_GLSp 9297",
  "50a8d45e-07fa-495d-8779-e07715b1c644___RS_GLSp 9358",
  "52acf240-2625-4b4c-8484-6c93e998b115___RS_GLSp 7337",
  "573ad2a6-906a-4d32-af81-0026fc57258f___RS_GLSp 4613",
  "584a0a63-7d44-469c-a0b0-f4d0aaa4c82f___RS_GLSp 4445",
  "58599ead-93e9-4ea8-a951-4ea380926f66___RS_GLSp 4649",
  "590078f5-c132-47d1-a575-9a05637cdaba___RS_GLSp 9351",
  "5958b15a-0ab1-411d-a5f2-99ef49572954___RS_GLSp 9275",
  "59726309-40b6-498d-a487-ff2d8fc455a8___RS_GLSp 4686",
  "5afd034f-f65c-4eed-9bde-f3dcb2950072___RS_GLSp 4422",
  "5b6fe4ff-01b0-4b9d-ad8d-4c6c8ef909be___RS_GLSp 9302",
  "5cfe5cf5-bdba-44b5-8295-c5bda6e34ae3___RS_GLSp 9353 copy",
  "5d69a1f2-cead-4661-8014-63e35e131850___RS_GLSp 4337 copy",
  "5f00e4d0-e48c-4b6b-9602-022fd38cf7e7___RS_GLSp 7331",
  "62e5f367-b111-478b-a187-88c298dc83be___RS_GLSp 9320",
  "636e39bc-897f-4ed7-8262-67f795eea324___RS_GLSp 9314",
  "648f5a4f-9fb5-486f-97ac-918c5804ab53___RS_GLSp 4370",
  "66151c80-6947-4062-bbb0-3771e6feefc3___RS_GLSp 7304",
  "671065f1-731f-45fa-9c11-3e0080f79ff3___RS_GLSp 4344",
  "6807988d-6cd7-4994-8469-8d48de642c2c___RS_GLSp 4380",
  "6db850c6-2b56-4acf-a283-2cd246c02bfa___RS_GLSp 4474 copy 2",
  "6df99d74-ad7b-4ca2-84db-904099c7293c___RS_GLSp 4321",
  "6feb357f-c0ff-484f-8c7b-5f4ec92f56cd___RS_GLSp 4338",
  "70f07cb5-244a-4140-99fb-0cc87e8914d5___RS_GLSp 4429",
  "716dd056-ffe4-4e09-a26f-6c90991b1aff___RS_GLSp 4288 copy",
  "71deb36b-bebd-4f7e-8ed3-362b1bbf86b2___RS_GLSp 4446",
  "7599513c-e2f3-4301-a010-6f32b429d047___RS_GLSp 4286",
  "76df2cc6-af5f-4f86-9dd7-d637936f9572___RS_GLSp 4609",
  "79aa374c-1687-42d4-bf48-84e8b1beb762___RS_GLSp 4408",
  "7aaf97df-dc18-4858-887d-e27dfe6c020e___RS_GLSp 7330",
  "7cc0887b-1bbc-49f4-8779-f7ff62aab90e___RS_GLSp 9354",
  "7eeafe66-9b88-473c-a2d6-a72edb99c3fa___RS_GLSp 9364",
  "806c6ae2-671e-425d-9625-604adcf5b46a___RS_GLSp 4289",
  "80a35c48-04ed-457d-a7e0-ac5179c7b53a___RS_GLSp 7292",
  "81a4c5ce-8c96-43e6-8391-415465688419___RS_GLSp 9274",
  "81babd8e-2bdb-4cea-baa9-9ae6d871aa88___RS_GLSp 9301",
  "86d512cb-65b5-401d-8b98-43cd03dbe815___RS_GLSp 9332",
  "87e9e4cd-55fd-4437-9f93-84cd25e744c2___RS_GLSp 4504",
  "88bd95d6-469d-4588-94c9-d11c330951bd___RS_GLSp 4468",
  "89a78c7f-dac6-426f-a0ad-397ce80e7a29___RS_GLSp 4289 copy 2",
  "89d03cba-77b6-4197-9993-434138cf6eba___RS_GLSp 7303",
  "89edb9c3-2ae7-4a95-937f-74778fc98df0___RS_GLSp 7287",
  "8bc4bad8-0337-4b33-858d-4d97aa0b6e4b___RS_GLSp 9366",
  "8da48548-461c-4574-9435-74310d8207a3___RS_GLSp 4387",
  "8ead1e08-eb49-4c03-b99f-f46f3597087d___RS_GLSp 4474 copy",
  "9103b8e5-919c-4d08-a282-25176874769c___RS_GLSp 4653",
  "910679a5-f18d-40ac-a00e-9e2c27a45610___RS_GLSp 9327",
  "919e2c3d-488e-440e-9841-60348cdb4adf___RS_GLSp 9322",
  "9210aee1-c3fc-4a6c-b98b-5b2634482985___RS_GLSp 4682",
  "966ea8fb-bc23-4161-810b-8c2251c6a1cc___RS_GLSp 4474",
  "97b35ced-b7ca-4f6d-8c22-5390c3d192c8___RS_GLSp 7307",
  "987f8819-fe72-413a-bf64-9a4d07acf353___RS_GLSp 9351 copy",
  "99238e07-5714-429e-9e3b-cdaebef9ab85___RS_GLSp 3810 copy",
  "9c78e47a-8ed7-4b8f-8230-499e2d135a0a___RS_GLSp 4335",
  "9d24f300-aaa7-413a-8abb-bf5b2b662fdc___RS_GLSp 4374 copy",
  "9e4b4ce4-514f-4add-81be-7ebafdacdb46___RS_GLSp 4490",
  "9e6ca942-064a-49ec-b7f9-8a1e25b5c0fe___RS_GLSp 4435",
  "9ffb3724-b220-4e79-9883-f1d3a5141152___RS_GLSp 4688 copy",
  "a0061e48-b26c-4610-933a-b4f4ce341707___RS_GLSp 7333",
  "a0c8342a-b9ca-4604-a2b0-7000bf7cdad2___RS_GLSp 4343",
  "a23f96ad-07de-46fe-b45d-bd5690db290c___RS_GLSp 4337",
  "a27ab77f-274b-4785-95f7-4844aa141c32___RS_GLSp 9353",
  "a401fce7-28ad-4d2f-bf98-3ef87ae65ba5___RS_GLSp 4614",
  "a47710cd-fdee-4562-8e91-1d7f405ca44f___RS_GLSp 4409",
  "a4937a5f-feb4-4d93-a75d-210a8d82bb0d___RS_GLSp 9296",
  "a5ab43da-2672-4993-b24a-46d0cae88fa6___RS_GLSp 9355",
  "a69da8d8-ed56-4bd8-a6ac-4208b8751cf1___RS_GLSp 9278",
  "a70422d5-9fb2-4d2f-9780-e341e8ea4c82___RS_GLSp 4380 copy",
  "a746eab2-b4d5-48f7-9dd6-9f9584a030f1___RS_GLSp 4328 copy",
  "a89fef58-2ae1-4fcc-ac6e-c9191a3ac3a5___RS_GLSp 9311 copy",
  "a95a00bc-f6af-4006-beda-81f8629cf106___RS_GLSp 9308",
  "a9bc0746-7cb8-4cf7-8998-27afed5648d8___RS_GLSp 7322",
  "a9eb57c6-db27-467a-aff2-253c58e26722___RS_GLSp 4446 copy",
  "aa261076-42a4-45b0-bc3a-e7fab382f194___RS_GLSp 4403",
  "aa2c6e9d-ce0f-4883-8b49-45412e96478d___RS_GLSp 7306",
  "aa7c5620-2ec9-4b2c-9091-8ce30597661e___RS_GLSp 9363",
  "ab76d00b-1082-49bf-9499-ca0640946ad9___RS_GLSp 9369",
  "ac871be5-33e2-4fda-995a-8818e8cb7fa2___RS_GLSp 4663 copy 2",
  "acbed711-885d-4f1d-986f-1fdf9b48cba5___RS_GLSp 7338",
  "acd57276-c222-4f47-b9df-c0728aad7183___RS_GLSp 9281",
  "ade34619-beab-41f6-8cb4-60b336be1cb6___RS_GLSp 7294",
  "aee8f52c-50c2-4327-8240-b8767ea8692d___RS_GLSp 9339",
  "b17b326d-8bb8-4cc7-bff3-183d2cc9a624___RS_GLSp 7280",
  "b2601912-33a1-4fb5-915a-d09d9808c7a1___RS_GLSp 7340",
  "b50e4d92-099f-4213-b5b8-0f76afa10124___RS_GLSp 4675",
  "b62826b2-1547-4c3d-bb4f-4ebc2e166b9c___RS_GLSp 4651 copy",
  "b73cddeb-5b44-404b-9ca5-6f5397f3d518___RS_GLSp 4400",
  "b9fbe0b3-7f05-4ec3-be16-0cf13b86e8fe___RS_GLSp 4374",
  "ba7f5741-75f8-4ac1-9107-ca73e244ac9b___RS_GLSp 9350",
  "bc1f8969-f563-4300-b792-93f75d23a483___RS_GLSp 9306",
  "bc690bb9-1611-4744-844d-0ab174467f54___RS_GLSp 4582",
  "bd9db5fe-e2d1-4c56-807d-2e90ea2fb6bf___RS_GLSp 4406",
  "bdfaa7d9-76a6-4e8f-86d2-f3a85d359321___RS_GLSp 4621",
  "c15dae11-38d1-47c3-99e0-c579ddeb3fb3___RS_GLSp 9269",
  "c29e2fb4-4bc4-4025-a29f-170a3477cd11___RS_GLSp 4317",
  "c4bd1ca9-533e-4a74-a4d1-8d633ac76438___RS_GLSp 4318",
  "c5776fbf-edd4-4dcb-889f-4e8971c73dd9___RS_GLSp 9330",
  "c6df65ae-6f54-4962-a7f3-cc21fd3e5b4d___RS_GLSp 3810 copy 2",
  "c7900124-4885-46fa-95b8-f335fbcd48d7___RS_GLSp 4290 copy",
  "c7b810f0-d451-4fe3-bc13-f1c89273e3e9___RS_GLSp 9331",
  "c7c98f44-a2a9-4c74-9cff-b0d31e5e05b1___RS_GLSp 4503",
  "c826221e-a999-4509-a7ed-d62726ca6078___RS_GLSp 4332",
  "c8769ca7-614a-4763-a8bc-8d7c7f1b2925___RS_GLSp 4355",
  "c98f6b90-ac72-4ea0-99d1-dfe5ff4d791a___RS_GLSp 4447 copy",
  "cc60499e-6802-47fd-b4cb-8ec70207e464___RS_GLSp 9305",
  "ccc55a8a-35d3-4621-88a0-0f81af01a7ad___RS_GLSp 3811",
  "ce28178f-e53a-4555-90b4-d7156fd8333a___RS_GLSp 9307",
  "ce3b19bd-dace-4d23-836e-233b02f12300___RS_GLSp 4505",
  "ceac7e26-3dbe-48a1-b111-3b4149cda69d___RS_GLSp 9316",
  "cfbb5e1c-2812-4467-9748-ee171462c331___RS_GLSp 4371",
  "d07122b2-20b4-4f1b-9071-bd90d77cc478___RS_GLSp 9310",
  "d0df634b-a2ee-4a22-8430-8a3b0e2c68c3___RS_GLSp 7344",
  "d1b2206b-39e5-4444-9ebe-223d0d78880e___RS_GLSp 4407",
  "d1f96a7c-c108-45a4-82f6-ffe84b0d081f___RS_GLSp 9304",
  "d39aadb4-c3f8-45a0-a92c-913263278c29___RS_GLSp 4654",
  "d46dd43c-2fdf-4235-bada-e7a2c5a04ec0___RS_GLSp 9273",
  "d4bf70c0-f7fe-4921-9944-a89717811183___RS_GLSp 4373",
  "d4c4842e-417a-4cc9-8fd5-40c618ddf483___RS_GLSp 4290",
  "d63f0a92-e489-4acd-b5df-847a83172abc___RS_GLSp 4607",
  "d703fb4d-1b5e-4fde-8cb9-d639a884e3ee___RS_GLSp 4401",
  "d7a856cb-b30e-4c26-9814-ce6ca9a9f188___RS_GLSp 4676",
  "d80bf184-8e5b-41db-a7a0-0784964e0fd2___RS_GLSp 4630",
  "dc846417-563a-4e11-b4ba-b093db6af0b7___RS_GLSp 4430",
  "de3ae60a-a0f9-45cd-b803-edb41b29a42d___RS_GLSp 7286",
  "de66a37b-002d-4e21-8f4a-a34976b74892___RS_GLSp 9283",
  "df225045-fc7e-4207-a8fa-072f0f262b8b___RS_GLSp 4674",
  "e4be1210-e9dc-44d0-8674-2dc7ecac0acc___RS_GLSp 4659",
  "e760af68-e885-46b4-a823-c4faf290c9d0___RS_GLSp 7299",
  "e90fe656-399f-4863-8014-03dc30e84f8c___RS_GLSp 4326 copy",
  "eba7c3d9-7e8e-4cfc-9a84-7eb39671d480___RS_GLSp 9318",
  "ec76caa7-e1a4-440e-9b4c-3ee355aac33b___RS_GLSp 4479",
  "eca81238-4c71-4982-a1a7-80c16852c299___RS_GLSp 4467 copy",
  "ed054cc0-58f1-4ca0-a369-2c04ffba43b0___RS_GLSp 4657",
  "ed2d6237-f67a-4a5b-bd44-f122388bcd80___RS_GLSp 4620 copy",
  "eda51a46-b237-47c4-b19f-97463e2c303a___RS_GLSp 4618",
  "ee8c711f-3aa3-45b8-95e4-c63f86929ab1___RS_GLSp 4444",
  "f435553d-4d3b-48a8-b13e-7639d330a001___RS_GLSp 4447",
  "f7044e72-6f61-45ce-9636-43f653381797___RS_GLSp 4637 copy",
  "f8101df7-db23-4241-9362-a00396806364___RS_GLSp 4336",
  "f8c850bb-1c2e-4340-8f00-bda707afcfa6___RS_GLSp 4451",
  "fa0d690b-b3ae-4ddd-b09d-ca8608233a1a___RS_GLSp 4629",
  "fadf58aa-98fd-402a-8fdc-d06e250ec265___RS_GLSp 4557",
  "fbcc9f67-8f83-4262-b7a6-26bf59d5fefb___RS_GLSp 4424",
  "fc90276d-e10b-4697-88b2-d9bbd87f622c___RS_GLSp 7290",
  "fce14144-8f76-43b9-9678-beb2984a56fc___RS_GLSp 4375",
  "fcf73483-d84f-4278-9ef7-5a715c73c8e1___RS_GLSp 4408 copy",
  "fd1e7be2-d635-4d70-85ac-f5d68ce7c2d1___RS_GLSp 4450 copy",
  "ff4e775c-b6cb-455d-b4c9-b635025078c2___RS_GLSp 4562"
 ],
 "Corn_(maize)___Northern_Leaf_Blight": [
  "008d9af0-7568-4a67-bb1a-0e915836ddc0___RS_NLB 4165 copy 2",
  "00a55069-3fa3-405b-8d87-4d3408a6ed98___RS_NLB 3645",
  "011b766d-7744-420d-a1a1-57bdff9b4ddd___RS_NLB 4128",
  "019e3ef7-aae4-4fd3-b95f-b4e914df7a39___RS_NLB 4284",
  "02504be2-4192-42f3-957d-df2841103562___RS_NLB 3955 copy",
  "028159fc-995e-455a-8d60-6d377580a898___RS_NLB 4023",
  "028a7e62-b520-42fe-8607-ee4d4bc6764d___RS_NLB 3844",
  "02eee796-1a1e-4d63-8247-a6aac6bfb272___RS_NLB 3546",
  "03c49c3c-6b9a-4253-b496-c9d5bcd819a9___RS_NLB 4249",
  "03c4fb76-81ce-4430-9927-9df7aad2c9e3___RS_NLB 3602",
  "04d8e1d6-766e-4fcc-9bac-42efb9e6ad33___RS_NLB 3620",
  "055934ae-8875-4c5e-8c7b-d72287ae967c___RS_NLB 3637",
  "057c3f91-c15e-464b-b4ba-66b16dbe5149___RS_NLB 3565 copy",
  "05eb5da2-11b6-4cce-9e01-724ca27cb99c___RS_NLB 4272",
  "060f9d88-a58d-43d9-9c97-90acc56627ff___RS_NLB 4227",
  "0654aea3-e038-4876-8088-30a1b693fe6c___RS_NLB 3574 copy",
  "0709f6e5-8e0f-47e0-af57-ab5cfcd1cb63___RS_NLB 4055",
  "071e0e62-6d68-4369-9723-d6e3f5f1545d___RS_NLB 3893",
  "07f5df82-5df0-446f-a13a-d35a2ef9b2df___RS_NLB 4226",
  "08afb40d-4814-481d-a0a7-760034a2e81b___RS_NLB 3657",
  "09140ec5-e485-4770-9c5d-dabd9727d6d7___RS_NLB 3721",
  "09160fba-83ce-4cec-887f-dce0abb79d08___RS_NLB 0832",
  "0a11f9e8-7357-48c2-8550-daeae59a1e76___RS_NLB 3588",
  "0a62fe5a-22db-42e2-bca0-53a8dcfd8129___RS_NLB 0810",
  "0abbec2f-123f-4ae1-a9b7-8babbe8d0e89___RS_NLB 3685",
  "0b002029-36c3-4ef4-b46c-29e45bf486d4___RS_NLB 0819",
  "0b112f62-54d5-4395-9f7c-d7ef05290be4___RS_NLB 4196",
  "0b3a35d8-25af-44ae-beed-398f591d5c1b___RS_NLB 0833",
  "0b4fb93e-1a74-42ae-9227-18d900c04bd7___RS_NLB 3670",
  "0b54aa70-56aa-4329-9e8d-b3c62c893f14___RS_NLB 4689",
  "0b5eb8db-3477-488b-9cce-6a7712d895ab___RS_NLB 4232",
  "0c28faf9-15f7-4e78-bb72-0a1035d95f1f___RS_NLB 4236",
  "0c3300b6-e4ca-401c-b34b-a9bf3dfa0a57___RS_NLB 3572",
  "0cf406f1-3004-4e13-87cb-e6bc02f67e51___RS_NLB 3566",
  "0da62220-7007-447a-8260-816c2b3afe7d___RS_NLB 3982",
  "0db9d2c1-46d7-4cce-aa6a-299ad6fd367d___RS_NLB 3608",
  "0e9f32b5-3423-4412-a9b6-69ba653db207___RS_NLB 3565 copy 2",
  "0ea6082b-93ed-4823-a7c5-acd907695d6f___RS_NLB 4080",
  "0fb76874-6a95-4071-87aa-aa1bfe63f815___RS_NLB 3979",
  "0fc1bc0b-2d7b-4c17-b32b-c7cc8301795e___RS_NLB 0829",
  "0fcc66f2-db0d-4b65-95df-a2e9c6d8f395___RS_NLB 4049",
  "1048748e-af83-48d0-ac54-2956ba4dc2e0___RS_NLB 4024",
  "1120da51-a165-47bc-b982-2a054f15186c___RS_NLB 3734",
  "11c62026-d2e9-425a-9374-563ef644e194___RS_NLB 4218",
  "11d96d18-2ccd-488d-a831-266871ee59b5___RS_NLB 4124",
  "1268904c-797f-4e33-bb41-afb58f369c6b___RS_NLB 3755",
  "13547fc2-2290-4be8-b212-3b96e0435db6___RS_NLB 4247",
  "1398ccad-700f-476f-ae12-fee3ccaa1931___RS_NLB 3978",
  "143ff440-2efd-4bc9-8312-265363c0872f___RS_NLB 3625",
  "14734668-ea31-458a-81dd-da6b177c4d26___RS_NLB 3694",
  "14c0cb10-f1bc-49d5-b03e-65a81805140c___RS_NLB 3826",
  "14db59c3-03bf-4d81-80b0-19e778ed17e7___RS_NLB 3542",
  "1517209e-897b-4ad5-b43a-8c9a8573c862___RS_NLB 4156",
  "15484387-5b5e-4248-928b-f5d4c98e8133___RS_NLB 3479",
  "15803dee-8071-49f1-81b6-fd3dedbc1c58___RS_NLB 4020 copy",
  "1600d5c2-b322-475d-bdf2-5609095bee27___RS_NLB 3499",
  "162b2f34-0fa5-49ca-8996-ef495e8ef6c6___RS_NLB 3484",
  "17cc72b5-9af2-4988-8b4d-f79a4054e26d___RS_NLB 3968",
  "180b911a-5c9f-4c76-972a-999e81d3b772___RS_NLB 3710",
  "18bfb36b-a816-43b4-8338-fe8793d91f04___RS_NLB 3665",
  "18c8ef79-bb1f-4021-aef3-1604fca63949___RS_NLB 3495",
  "1955cc33-ecd0-4853-9743-3620d815bbfd___RS_NLB 3577",
  "19b752b4-9c83-48ab-96dc-d6756644680a___RS_NLB 3581",
  "19d09715-275b-473b-a1e5-4febed1421db___RS_NLB 3703",
  "1a11d38d-9be6-425c-bc0a-c0f158d6f5d8___RS_NLB 3667",
  "1a367958-51c4-4ebf-86ae-d6a5dc7e1114___RS_NLB 0770",
  "1a6b22db-27fe-4c78-b014-64f1879976dc___RS_NLB 4062 copy",
  "1aab2df1-0705-4ca5-a93a-42e93aa240f4___RS_NLB 0804 copy 2",
  "1b0ee938-dc5b-44a2-9d22-37aae22aada4___RS_NLB 4129",
  "1bb3a6f8-06a6-4325-946c-b6bf11eac3c1___RS_NLB 4012",
  "1bd14889-1157-4e89-854b-c4078917ae23___RS_NLB 0805",
  "1d5e0e56-5aa0-4c7b-9edf-e7a51110e6c4___RS_NLB 4113",
  "1deab1d3-6e84-4b2e-95cd-1f907660e59a___RS_NLB 0791",
  "1e880696-d92b-4dbd-83d7-9ca9f8f00965___RS_NLB 4275",
  "1e974f0a-834e-4288-bba8-134c860882f5___RS_NLB 4268",
  "1eb43526-92fa-4fbd-b09c-899511d7a79c___RS_NLB 4039",
  "201bc21a-5372-4c91-a263-ea36ed96665a___RS_NLB  4628",
  "203e38d5-5a1b-463a-85b8-cba4aa1da1b6___RS_NLB 0806",
  "209d61fa-710d-4b89-8f1e-83d3dff44c05___RS_NLB 4097",
  "2144ae7f-b1b6-4f09-ab5e-954db9df1f6b___RS_NLB 4053",
  "215a8c48-d6ef-4751-b23b-800bc8692ca6___RS_NLB 3503",
  "21e76128-1306-4a69-a0ff-60d4a824040f___RS_NLB 3661",
  "2270041a-e14b-4b57-9906-c5a3991a1b09___RS_NLB 3556",
  "22d215df-2361-481f-b3a7-27e3e2d5cebb___RS_NLB 3587",
  "230579e7-a090-41dc-bc71-e796d898523a___RS_NLB 3616",
  "235cb141-61ea-48ef-b556-4ea7453d5ca3___RS_NLB 3731",
  "23f7ce7f-23fc-42b1-86d0-69c7f7010812___RS_NLB 3639",
  "24074ddc-4302-4dab-9cd4-c34e3f071dc8___RS_NLB 3836",
  "2418712a-bebb-45ca-b632-4f0548133221___RS_NLB 3477",
  "246652d1-e8a5-4dc9-a252-6199fcc5fb34___RS_NLB 4221",
  "248e2b4f-4a50-460a-8ef4-cc8fe3c8bd12___RS_NLB 3872",
  "24da71bd-3323-41f8-a6e2-7d1a4a438608___RS_NLB 3537",
  "24fee780-a78e-47d7-a48f-604dadf295ce___RS_NLB 3498",
  "253f721d-33bf-4c54-a267-277965e95257___RS_NLB 3633",
  "2646db6b-dba9-4de7-b569-adbc96d3f127___RS_NLB 4269 copy",
  "267d9755-8549-46d6-b6f2-270b70ebe5f9___RS_NLB 4059 copy 2",
  "27aa7fa8-c66c-45b6-a57d-96a14e05af99___RS_NLB 3564",
  "27fcbe5e-c444-4516-b936-c95871a2e2c4___RS_NLB 3533 copy 2",
  "281807e5-a1db-4bff-88f9-dbb7535a1b53___RS_NLB 4177",
  "297448cd-10e7-4f57-bb1c-ae02d6134565___RS_NLB 3849",
  "29955528-2271-4b76-9ca9-99a48405f40e___RS_NLB 3603",
  "2a1f3a15-26be-4a24-b62e-e47626e462ef___RS_NLB 4203",
  "2a4b3fe0-6858-459b-b620-cd01d0cd6333___RS_NLB 4273 copy 2",
  "2a5c2a7a-c078-43ce-b50b-8a4f33049415___RS_NLB 3474",
  "2a5d32e1-5b7f-4fc0-8078-87fd4de96375___RS_NLB 0794",
  "2afe409f-5bae-46f0-bcec-9eeb69ecd12c___RS_NLB 0812",
  "2afecd32-d0a3-44d2-b7fb-daa25989701a___RS_NLB 3524",
  "2b2f19f2-757c-47a0-a519-4cec0095a840___RS_NLB 3708",
  "2b7de414-15f7-44ef-aef3-12e75819ac31___RS_NLB 4188",
  "2c2dd54a-38b2-4b9b-833a-52c5ed1ccede___RS_NLB 4050",
  "2c827454-4902-4ff2-9237-687281600a19___RS_NLB 3736",
  "2e1013a9-88bb-494f-9ed5-7e8e79f9a7ca___RS_NLB 3606 copy",
  "2e594c45-c4c7-43b1-ab19-100945e7a2a7___RS_NLB 4020 copy 3",
  "2f266f09-ce50-4cc6-b2e5-5846e3595bd4___RS_NLB 0760",
  "2f80ba76-4fdc-41cd-9b6d-4b1255a29b83___RS_NLB 3687",
  "2f91a397-315c-45bb-9d88-3d1389c82d98___RS_NLB 3508",
  "2f9acce2-8616-4cd3-8aac-b57cd9a03084___RS_NLB 4228 copy 2",
  "2fd737f7-bba3-4df7-8bd9-8c1d78b96bf3___RS_NLB 4127",
  "30f2767e-d618-41c6-ae4f-d7ddf806318b___RS_NLB 4068",
  "3103f262-0222-41f6-9ce3-8f18e8aa855c___RS_NLB 4243",
  "311621c2-ce96-4e4d-ae49-b303619e1f9f___RS_NLB 3632",
  "3138c754-7a79-4d77-8556-edbbf78184d4___RS_NLB 4084",
  "3146c9fd-b7eb-4e06-b063-6d9de01a645f___RS_NLB 4222",
  "3164b99c-5a51-4a73-8716-8e2f6548bf8b___RS_NLB 3541",
  "31a49b40-0f0a-496d-a3bc-cf566bbbe845___RS_NLB 3568",
  "31a6d935-89b5-4feb-8fc6-44300ab3639e___RS_NLB 3640",
  "34ae03ee-4314-47b1-a568-f3537bc2afba___RS_NLB 3672",
  "34affa73-9797-4d0e-9cd6-916f8a6c3df3___RS_NLB 3521",
  "34e1850b-628c-46eb-b8b0-9618025b1f84___RS_NLB 3729",
  "3513a96e-7db5-4a2b-9dd8-79847c92e75b___RS_NLB 4045 copy 2",
  "369b969a-f820-40ec-a43c-7500df90e6d1___RS_NLB 3615",
  "36f36ace-ecdb-49af-b1ec-a29d8b0ae918___RS_NLB 3836 copy",
  "37665d4b-1063-4929-b9e1-88fb5439cdfb___RS_NLB 0765",
  "37da66ff-ddaf-4848-a9df-f48bd43da831___RS_NLB 4214",
  "381bb492-a362-4423-8e51-f59050a82d83___RS_NLB 3981",
  "38ee65f9-32f9-49e7-83f8-277ad34fff0d___RS_NLB 4090",
  "39204770-c7aa-4ca0-bd41-2acd39cb699b___RS_NLB 3712",
  "3949d36a-42a7-4d40-a096-0b7d32698b04___RS_NLB 3502",
  "3966eebb-e814-4cc8-90c8-2f91fa5d5ba9___RS_NLB 3897",
  "398fe3cf-ca89-421f-ad3e-b16c720835c3___RS_NLB 3723",
  "39adac5b-ad7c-4ebd-b1de-a583229cc845___RS_NLB 4244",
  "39bbce49-e9e0-463b-8c87-a3c0b1204050___RS_NLB 4011",
  "3a0c570b-dfd1-4681-97be-658151e7279f___RS_NLB 4099",
  "3a29a411-e364-4a88-8c32-14051876ec60___RS_NLB 3713",
  "3b8446bd-30df-4ff8-87e3-bcddae3b4e7a___RS_NLB 4108",
  "3cb5b2ea-1eb2-4b94-9300-2aceb0863d75___RS_NLB 4138",
  "3ddba6ba-0467-45e1-b3d7-24b97ee2c767___RS_NLB 3720",
  "3f494938-cfae-4d4e-9cc6-d7fdb15429ea___RS_NLB 3530",
  "3f7ded82-d9d3-4b3d-9beb-58c5a43f894c___RS_NLB 3656",
  "3fdd99f1-a3df-4498-8b6a-b844621a72ca___RS_NLB  4670 copy",
  "40c2375e-92e1-4259-b268-d2433137e456___RS_NLB 4009",
  "4218c656-20b5-46e9-a24a-d69e6ceafb05___RS_NLB 3548",
  "4266e4a0-2364-4d9b-9dcc-9f4a5b60a5ba___RS_NLB 3658",
  "4284aa42-8c36-4674-8ecd-70a7eb6ddd80___RS_NLB 3730",
  "429213c9-31f8-49e0-bf27-5ffe7c43100d___RS_NLB 0802",
  "42e5960b-f56e-430d-b729-e48a543a136f___RS_NLB 4072",
  "431a8c6c-3eed-4079-ba49-5bc7a12b94b0___RS_NLB 0766",
  "432d152d-dd08-40b4-ba2f-2cbf2352d29c___RS_NLB 3586",
  "4376c828-f03e-4953-a2db-bd6405eadc68___RS_NLB 4061 copy 2",
  "45b08cd4-63ef-428d-b220-3b8bb147291b___RS_NLB 3679",
  "474797d0-3298-4480-ba05-7c49026ffbbf___RS_NLB 3529",
  "4750f625-023b-46c8-8236-9a3c6bb93c73___RS_NLB 3471",
  "476e3e28-06e0-47ba-980c-4eb5e1fbbfe9___RS_NLB 4235",
  "47a01d34-e691-4337-8d17-cbe8ee34b843___RS_NLB 4164",
  "47e485da-d2e2-440f-ab6a-b17bcb0a83cb___RS_NLB 0808",
  "4832c7a2-532b-4d5e-9949-fd295d171cfa___RS_NLB 3660 copy",
  "48d6bed0-2514-4d2a-b207-ebff452a62a8___RS_NLB 4269 copy 2",
  "49093468-8fcf-4adb-9787-610b6b216b4b___RS_NLB 3692",
  "493500a1-d3d1-4cf9-b354-78540b3e3044___RS_NLB 3689",
  "49d8ec4d-4600-4f2e-a04b-01dabb76aa21___RS_NLB 3735",
  "4a8adf2a-73d4-4555-8621-1292de40b194___RS_NLB 3547",
  "4af2c376-1e04-4649-85f2-a2c69463388e___RS_NLB 0759 copy 2",
  "4c640c66-bb10-4f7c-ae15-6be2b885c6d2___RS_NLB 4006",
  "4d669c56-5dff-4b5b-9395-85fd4ec204fc___RS_NLB 4266",
  "4f18b762-1a20-4367-bce3-41a58620a6d2___RS_NLB 4269",
  "4fbae838-e991-4294-9126-7ab7d7838349___RS_NLB 4046",
  "4fd7db79-097d-4344-a4fa-3657421fe654___RS_NLB 0800",
  "4feaacb9-a17f-4b39-85d5-85e5fb9997a3___RS_NLB 3575",
  "505ce513-98a9-4100-b7a0-f0767eb598c8___RS_NLB 4206",
  "506d1785-00e5-476a-94e1-a3e96dd22cb5___RS_NLB 3739",
  "50f917f3-8a84-4379-adc3-ee0501043f4c___RS_NLB 3718",
  "516ebf4e-2572-4ba9-aefe-4d11fbb374b5___RS_NLB 4082",
  "520410b0-012a-4c6e-af78-a6971ddfb612___RS_NLB 3835 copy 2",
  "534e86f0-1e17-4fa0-b042-b0b26ab53eb4___RS_NLB 3745",
  "5376d36b-07e3-42b8-9b5f-1ec2d9e720ed___RS_NLB 3684",
  "5388d277-8739-49ac-9d60-f842621fdeac___RS_NLB 3682",
  "5400a487-4b38-40d1-bfb0-7d9e89f58de9___RS_NLB 3610",
  "54cac376-96d8-4029-91a5-935cd1c8d24b___RS_NLB 3845",
  "571d15d7-e0eb-43f4-aca7-abcdcf1e80a5___RS_NLB 4213",
  "57d6fe49-6ac4-4672-ae90-02729f1430fb___RS_NLB 4197",
  "599ee05a-0659-4159-a65f-43278e6f8f55___RS_NLB 3841",
  "59bc212e-6b39-4712-b9c9-6edc3fa3e05f___RS_NLB 3604 copy 2",
  "59d0b355-5722-4c98-9498-83264f1e9a84___RS_NLB 4020 copy 2",
  "5a2e998f-6157-42a7-a2b3-bfb972a8c94e___RS_NLB 4111",
  "5a341520-f90d-40eb-89a7-e557aa572200___RS_NLB 0764",
  "5a6c652d-93e0-4de5-8214-caaadb09c840___RS_NLB 3848",
  "5a7153bc-6cad-4b5d-8ee1-23fdba51ab2d___RS_NLB 4219",
  "5ab0a55a-509c-4c16-8e16-cb2880f0677d___RS_NLB 4105",
  "5ab44c38-1773-4f3d-bc14-cc1ccf8c88a6___RS_NLB 3512",
  "5b08c1fc-7f88-4537-acbc-d5924f74f865___RS_NLB 3636",
  "5b4f7b36-0ce5-491c-93bb-ac81b21b13c6___RS_NLB 3824",
  "5c339413-2b34-4c09-8343-c88e216a29eb___RS_NLB 4193",
  "5c867753-033e-4b36-b7dc-bc6470842130___RS_NLB 4233",
  "5c9325f7-10c4-4628-a6e5-f5932f1d4122___RS_NLB 4261",
  "5c9adc70-1f97-4b53-a7cf-149c5c98664e___RS_NLB 3663",
  "5cb67eaf-e897-4fa2-91d3-83f7175918da___RS_NLB 3673",
  "5d8533b8-c57e-49af-aa9e-e4ab9f28f92b___RS_NLB 3691",
  "5df7a420-aeab-4eb0-959a-290e4bb2a944___RS_NLB 3514 copy 2",
  "5e1b2b72-fe1d-4bf2-a5ed-b3a013ffabe9___RS_NLB 3696",
  "5e324c24-0a77-4f76-9870-b320acb4b265___RS_NLB 4126",
  "5ea1de01-683f-4080-a341-6eafb3d41bc4___RS_NLB 4154",
  "5f1fc34e-15d7-4529-b136-3c35c12bf645___RS_NLB 3648",
  "5fc48a9b-7aee-4262-adf3-1102e354dd61___RS_NLB 3690",
  "5fe9d333-4801-4913-a1a2-aedb25def4d0___RS_NLB 3714",
  "60fb1e70-5368-4b1f-ab46-12c5add010e6___RS_NLB 3839",
  "6150b5d0-047c-4f46-8832-9760d8359730___RS_NLB 3669",
  "61d2e3ab-91a7-4bbc-93a2-c2376b3c434d___RS_NLB 4264",
  "623dcf1e-68f1-4de1-a69c-c94b23ec11cb___RS_NLB 0776",
  "633dc7b2-959b-4537-a9fe-eb88067611f7___RS_NLB 3996",
  "63d75f8e-ad04-46d0-bc14-9c40a6961a80___RS_NLB 4185",
  "640ff351-7941-4dd0-8276-d2325150dfad___RS_NLB 3513",
  "6522ad4a-5629-4510-ac5e-bf854c42d3fb___RS_NLB 4034",
  "653472ab-cec4-4253-a85e-6d1318f1c15f___RS_NLB 4054",
  "653a38de-840f-4905-83f2-34a96410bb99___RS_NLB 3853 copy 2",
  "6544d35f-ffb9-4a00-8240-c6f50c4e5e41___RS_NLB 3500",
  "6578fc09-44ed-4f4b-8ba9-d420a169e824___RS_NLB 3983",
  "66045b64-9ade-42cc-b572-2683245d6816___RS_NLB 4074",
  "66abbb46-1197-46c1-984e-0141246142b8___RS_NLB 4245",
  "6728afe3-9ab4-4235-b7e3-5b9ee80b6675___RS_NLB 3605",
  "67352cb5-36c2-4548-99b6-4138741f0b08___RS_NLB 0792",
  "6868de8b-1a90-448d-93e8-265f2fa401dd___RS_NLB 3929",
  "6880b763-1f12-4d80-9aa8-a2b8eecaed1a___RS_NLB 3543",
  "699bf6c9-fe54-4f33-84c2-559329820008___RS_NLB 4083",
  "69ac9833-5f27-4fd4-b117-8ca2de2a2db3___RS_NLB 3482",
  "69ebacf6-aff4-40d5-82de-95624c131503___RS_NLB 3526",
  "6a43e0eb-5366-4170-9782-12c16a853d04___RS_NLB 3519",
  "6a72d38e-1bde-4d45-892d-891574de8ca1___RS_NLB 3576 copy",
  "6aa24d2e-df4b-495b-a246-91f14df01f68___RS_NLB 3516",
  "6b1af2e2-fa9a-49db-8986-22f9a4472d01___RS_NLB 3744",
  "6b2ebc67-09f1-4d57-bdb4-65f5f5a6c7a2___RS_NLB 3876",
  "6c8c49d0-1730-4829-aa21-ccff3079881e___RS_NLB 3540",
  "6c9054a8-bdcf-4e48-8baf-cc4638b9cea1___RS_NLB 3747",
  "6e9d79b9-3d43-4e52-86b9-e131d62407ec___RS_NLB 3743",
  "6f129430-750d-4d02-b509-1ec760ae8aaa___RS_NLB 3486 copy",
  "6f370eac-7a90-40a3-be93-62147f32041c___RS_NLB  4608",
  "6fd2d660-7294-4a2c-8626-27228b1c8b70___RS_NLB 3627",
  "72588a71-5b19-4867-a1d4-1a3addca8ab4___RS_NLB 3553 copy 2",
  "7272d46b-b201-4ffc-8634-1555b87e23ff___RS_NLB 3472",
  "72b09e95-0595-408e-8167-d765d21c6766___RS_NLB 3585",
  "7325478b-3516-4cb0-8635-57f785b6262d___RS_NLB 4109",
  "73a8b159-a494-458e-8177-225cf3cb4618___RS_NLB 3846",
  "747bb71f-c1d5-47d7-bb5c-4dff1f1b3c9c___RS_NLB 4136",
  "74c8a3a5-7a9d-40a4-9ec7-4003ef2a6eed___RS_NLB 4251",
  "751475e8-516b-4708-8509-7137931ad7ae___RS_NLB 3597",
  "75820bce-be66-4da7-ba84-987bafe3afc5___RS_NLB 3752",
  "76773e2f-9cd1-46d1-aeb5-f6481d3066ee___RS_NLB 3998",
  "76ac9115-7016-46bb-93a2-cbc420792406___RS_NLB 0782",
  "77179782-2d6e-4056-ad95-f8263be229ab___RS_NLB 3674",
  "772c92e7-3535-4f03-a85b-67b843123079___RS_NLB 4125",
  "77798831-63a1-44a6-81ba-a86e673b9cf1___RS_NLB 3489",
  "78d28bb5-2b07-42eb-aff8-d205eaf38fc9___RS_NLB 3967",
  "79d78748-a6b6-489f-813d-5bceb0e9988b___RS_NLB 3666",
  "79dc5a4e-60d1-4185-add7-0abca4a2205e___RS_NLB 3549",
  "79e82d45-5edf-4143-b21c-f60f560ff75d___RS_NLB 3614",
  "7a044a04-7cfa-4677-aaa1-52f3686a5a2c___RS_NLB 3594",
  "7a1e1c85-a9e3-4520-9349-8e374740c40a___RS_NLB 3753",
  "7ad6bd48-7732-4487-8182-8b1d11b10a07___RS_NLB 4270",
  "7b45e9ce-9199-4f4f-9831-ef52e7c61816___RS_NLB 3547 copy 2",
  "7bfceeb5-366f-4cdb-8b5c-9bde0bab6b01___RS_NLB 4010",
  "7c3b05fe-d387-4542-8a0b-d027d7c9a72c___RS_NLB 0821",
  "7c9fff54-416b-49f4-8afa-bc8e1980e02d___RS_NLB 3583",
  "7cb45f15-0315-4cb4-88fa-c0bcef1a5a73___RS_NLB 0777",
  "7d55a6f5-5101-4103-b34c-5e3d764d2757___RS_NLB 3592",
  "7df4e119-8ef5-43db-a3dd-7e9a0303eff7___RS_NLB 0799",
  "7f308111-9ef3-4ee0-b820-0af9743f8040___RS_NLB 3547 copy",
  "7f5a02de-b04a-4476-8bfe-dc0980913892___RS_NLB 0804",
  "7f6ca3f9-1023-45f1-9f1c-bd530355367f___RS_NLB 4175",
  "7f79300d-eb62-4e8d-8eeb-6c6097afea01___RS_NLB 4045 copy",
  "7f948f93-322e-4ef4-b43f-1b17d83ea983___RS_NLB 4041",
  "7fb389d4-38a3-49de-8f68-72a87ff4ae0a___RS_NLB 3716",
  "7fc13d9f-316e-41ff-b216-5665c08357e6___RS_NLB 0784",
  "8084d22b-8026-4f2a-9820-2e4f4aba659f___RS_NLB 3520",
  "8115e99a-13a1-4d49-91aa-15189fea7a28___RS_NLB 3606",
  "81194232-6686-4910-8fb3-f5e2ec369684___RS_NLB 3984",
  "8152eff1-a36b-4377-96f8-424ff6a97f35___RS_NLB 3727",
  "8164afe9-7944-4e39-a7e8-f84904c09715___RS_NLB 0780",
  "8187d1cd-0de8-48d9-bd90-48c718ce090d___RS_NLB 3702",
  "823df39c-56d2-4a28-b757-61d19db5152c___RS_NLB 4235 copy 2",
  "824f1625-2e71-4df7-ab63-e0093d9a46a0___RS_NLB 3850",
  "828594dc-1ed5-4b8b-84a4-90e0d7225969___RS_NLB 0790",
  "83327a8d-9bfd-49d6-9fc9-bc065a379bce___RS_NLB 4044 copy 2",
  "839ba99e-e297-43b4-9067-69b74ab61ab7___RS_NLB 4063",
  "845ece89-ec44-4727-b5f4-8df8407b6fea___RS_NLB 4039 copy",
  "85f5a1b1-18bb-492e-834b-b908bebd3607___RS_NLB 0772",
  "869f3ccf-c9f3-44f8-8214-4a513d4eb2df___RS_NLB 0779",
  "86c4280e-2954-4d0c-b9d5-711cab634904___RS_NLB 3641",
  "87127781-0ffa-401c-8a40-476533092885___RS_NLB 4667",
  "87705bdf-957a-4abb-b8b6-159d64de4e02___RS_NLB 3985",
  "87ae7169-5770-493e-93c2-761f6e885bad___RS_NLB 0816",
  "8808dc45-6683-4e50-a324-938dbb78afc9___RS_NLB 3754",
  "8970b98a-13dc-4293-8795-33436cef05e7___RS_NLB 3559",
  "89faef04-7b28-41bc-a910-d89890f028d1___RS_NLB 0795",
  "8a936e9c-617d-4f4c-834a-6b4cb40db290___RS_NLB 3660",
  "8b09fccc-2ee5-41dd-ac51-99e82e2f81f6___RS_NLB 3518",
  "8c22c962-864b-4d20-9b94-660407f90533___RS_NLB 3619",
  "8d420db3-e5d3-4537-a3f9-d57263843f21___RS_NLB 4073",
  "8e0669bb-1959-4f28-b98d-da3c2d85396c___RS_NLB 3678",
  "8f0fc373-ecfa-4420-bd93-d8693409dc3b___RS_NLB 4021",
  "8f633697-a90c-45ed-b056-8c69f3e32eaa___RS_NLB 4265",
  "8f9ac1a8-f91f-4895-ba02-a2ff696fa43e___RS_NLB 3486",
  "900c8aee-0d1d-4963-aa84-42d9be09b665___RS_NLB 4112",
  "9135c535-4123-4e5e-af3f-98ba6723c678___RS_NLB 4078",
  "92274867-1466-479b-81df-23eac6437781___RS_NLB 4047",
  "92d654a4-bd4d-48ed-9bdf-1750662ae4af___RS_NLB 4088",
  "92e0ba47-165f-457c-9be0-baa37eb16d40___RS_NLB 3624",
  "931b2ec6-2ed7-4a70-8309-7bce6a01d0fd___RS_NLB 3757",
  "937bf586-316f-4916-9a51-c9addfc9ff2a___RS_NLB 3733",
  "948adb0d-8641-48ee-ba43-53e34b33bece___RS_NLB 3590",
  "952a1717-81d6-482d-b13b-3897a420fd47___RS_NLB 3994",
  "95b2bcb7-56f0-47e0-82d1-7328e6863de9___RS_NLB 0787",
  "95eedb2c-7ebc-438c-82ad-99d2009e7f6a___RS_NLB 4077",
  "9634137b-5b5b-49eb-875a-82da77a5bc54___RS_NLB 3697",
  "96e81d2a-1f67-423a-905e-6b0b3543f120___RS_NLB 3576 copy 2",
  "97bb609b-99f9-4dd8-9468-cb24cb17bf7c___RS_NLB 3497",
  "98c96eff-fa58-48bd-a2ae-3b2f2bfa7989___RS_NLB 3751",
  "98f774a9-df98-45d1-903a-b0c4f6fe6b59___RS_NLB 4211",
  "9943cc83-f708-487b-b45e-a9ac23ee3973___RS_NLB 4045",
  "995678f2-e77e-4db2-a858-49cfc0f2d423___RS_NLB 3609",
  "99ee381f-c18c-4cf4-b62a-af315af69d5d___RS_NLB 3493",
  "99f77104-ed40-46c6-aed2-91c58cc3ffa7___RS_NLB 4246",
  "99ff5228-10ed-433b-bb50-1a1c7869a58d___RS_NLB 4253",
  "9a16ff5e-7227-4739-8e42-f1bc3e478acf___RS_NLB 3699",
  "9a4feadf-4940-4c42-9dd8-18f0e98c4c58___RS_NLB 3539",
  "9bc5549b-6193-4d07-961c-e9a3a3f639ba___RS_NLB 0788",
  "9c75d1f3-9a11-4b76-90cc-ade68ea91ce9___RS_NLB 0824 copy 2",
  "9ce41fa1-3d14-4b66-9a13-da488a8727d7___RS_NLB 3997",
  "9d5c72fa-9b60-45fa-aca8-98db5ef39f0b___RS_NLB 4098",
  "9dffdf42-2221-4cab-8228-35b011956849___RS_NLB 3567",
  "9ebe7867-1d2c-4d72-8a7f-ff45ea5fa534___RS_NLB 3672 copy3",
  "9ed2cd87-1fa0-4425-bd07-d09cae22a936___RS_NLB 3525",
  "9f0cbfae-ce1e-48c7-8fdc-496c57e4d9d0___RS_NLB 4153",
  "9f5b5f44-b2cb-40d9-87ba-46376528f25e___RS_NLB 3852",
  "9f9bba25-9ffb-472d-8fb7-b887ba59de51___RS_NLB 3980",
  "9fec4d0d-462f-4b8f-8731-be6961908639___RS_NLB 3612",
  "a0d31f49-895a-4921-afb7-a546f6589433___RS_NLB  4670 copy 2",
  "a1239833-7c07-4f71-b81a-44bbdf370f1a___RS_NLB 3825",
  "a25e0982-60a0-4282-bedf-0512171e2fde___RS_NLB 4061 copy",
  "a294edd3-2e37-4901-b35a-454480f4cea1___RS_NLB 0768",
  "a2dfefdf-9983-46a3-b6b2-8a0aebbf390e___RS_NLB 4147 copy 2",
  "a3155868-d50e-45d7-a0bf-870d3f5d16eb___RS_NLB 4263",
  "a3359351-66b0-4774-8165-6e002c8fb96f___RS_NLB 3634",
  "a34e16c2-cab4-43ca-9ddb-b877ea3228b7___RS_NLB 4189",
  "a34edad5-bf33-4d6c-bf2b-68a35c3f6987___RS_NLB 4181",
  "a3597d3e-d2db-4068-aa85-d09464197fe2___RS_NLB 4137",
  "a3a57aa7-6785-48a8-9bb6-f2726d365f9f___RS_NLB 3686",
  "a3b588f0-1c34-4a1e-ad47-c2202a896173___RS_NLB 3834",
  "a3be7c5c-5a5c-4b66-87f9-d74a9d5d409d___RS_NLB 4228 copy",
  "a3d50e22-77d3-4bde-a749-0b9f3d7448f7___RS_NLB 0796",
  "a4feb7a4-f0e0-4e95-8756-a50e893a0436___RS_NLB 4103",
  "a523ac3f-ae53-4da7-a3e1-7cce38714f75___RS_NLB 3812",
  "a690d434-444c-4e06-bb9d-6661e8c6ba89___RS_NLB 3598",
  "a70d7087-a41c-4f67-a6a6-0f48c8bd4700___RS_NLB 3709",
  "a7a2af2d-ec31-4567-ac30-e6cc593e1cb9___RS_NLB 3995",
  "a7a738f0-39e4-4541-828d-92c5dd42ee86___RS_NLB 3544",
  "a87efc8b-924b-4b99-820a-c4738bbdc446___RS_NLB 4259",
  "a895ebbb-d81d-4900-b75c-740407e5bf6d___RS_NLB 3510",
  "a8eba89e-be4a-4976-bf12-8386f0caa338___RS_NLB 3740",
  "a974dc85-29d7-4441-afa1-06a47d12df6a___RS_NLB 3629",
  "aab5319c-f249-41e2-86c1-ce96c4c89ca0___RS_NLB 3504",
  "ab2c2346-36e9-46ee-8762-2444560774eb___RS_NLB 3532",
  "abb52b8e-2034-4060-8f45-9d5220cd0906___RS_NLB 3931",
  "abd4ee1e-822b-4ff5-a8f8-2a7a27aa6584___RS_NLB 3517",
  "ac38d408-9cc0-4979-966e-b4bf6413e84c___RS_NLB 3732",
  "acb55b5e-ed74-48b1-8853-0f44e943abe0___RS_NLB 4104",
  "acfc653f-44e2-41f6-85cd-3595a6fd7455___RS_NLB 3840",
  "ad159032-2941-402b-bd42-5d79bd3fa9a8___RS_NLB 4116",
  "ad75b9e5-d8c8-4719-8bfa-61bad83ff706___RS_NLB 3565",
  "ad8b4ef5-2f8c-4cef-819e-cbe0ec19ae17___RS_NLB 4199",
  "add115ab-8070-4694-9aa7-6113253da96e___RS_NLB 3706",
  "ae554dde-24e0-44df-82f1-5e4a1b3dd86c___RS_NLB 0775",
  "aec48247-935f-4062-a1ba-7018ee81f5a7___RS_NLB 3579",
  "af44d1ff-8fae-49a0-ad5a-2b08204e397c___RS_NLB 3722",
  "afa9db47-da6a-4914-b218-5612baf34d87___RS_NLB 3501",
  "b044fcc0-9215-42c8-beac-4e67b1e7eee0___RS_NLB 3523",
  "b05a1bee-0b31-4c32-8091-8257ce42abd0___RS_NLB 4207",
  "b289a3cb-17e1-4889-8b12-bbee01d7b4e8___RS_NLB 3724",
  "b2dadd42-82b1-415a-8305-d6a720707912___RS_NLB 0771",
  "b2f77b0d-b331-4fa6-81b1-fa5d56674a64___RS_NLB 4076",
  "b3828c2f-54ca-4673-91e0-3c531a5a8be5___RS_NLB 4282",
  "b3e0815f-d7dc-4481-850e-53d917c244d7___RS_NLB 3569",
  "b4c32f6a-4be5-463d-9616-48f4d999a310___RS_NLB 3693",
  "b525d51d-7e0b-4999-ae36-272ddd899fbb___RS_NLB 3737",
  "b5721e14-a2cf-4010-96cf-9716630155e1___RS_NLB 3552",
  "b68a6a63-8711-4f14-a5c6-ddcd8a2cae3e___RS_NLB 3847",
  "b7494cf3-21b0-4673-9df9-18841854c3af___RS_NLB 3930",
  "b761537f-f90f-43a6-a816-6d515dfa0e26___RS_NLB 3797",
  "b7ec2193-2a49-43b9-9962-7a13d472e668___RS_NLB 4044",
  "b7fc574e-e12e-4dc5-94c2-698892884e66___RS_NLB 0817",
  "b8313909-4b3a-4d5e-9d9b-616e5749bf3d___RS_NLB 4086",
  "b84feecb-e6b0-44e2-a1eb-7be061ac28b3___RS_NLB 4079",
  "b88f09d6-6951-48bc-ac61-ca9cc9e988fa___RS_NLB 0828",
  "b8b9d76a-31a4-4e15-b96c-3a22a9f8c40a___RS_NLB 4231",
  "b8dc4415-0b18-441b-af91-2d7741539b5d___RS_NLB 3514",
  "b927fa6f-d9c6-4fa0-b93f-ca77f7cec655___RS_NLB 3827",
  "b9aed562-345f-4010-b898-33359c4c85db___RS_NLB 4202",
  "ba97ee35-01d7-4cb4-a44c-913c329ca328___RS_NLB 4149",
  "ba99509a-8a3b-48b5-b298-748ab3c2bb63___RS_NLB 3719",
  "bacc8c75-6b9e-4d50-a4f2-4c543b759b71___RS_NLB 4273",
  "bb45a9ae-144d-4aa1-a863-76969dc9a84b___RS_NLB 3527",
  "bc0f795a-8434-4184-8564-8e9c9e642356___RS_NLB 3578",
  "bc6c495b-3b17-45fa-8ff6-2e8bd528c5a5___RS_NLB 3562",
  "be7a6052-bb7e-4348-bc97-5591277fa769___RS_NLB 4273 copy",
  "bf03fb8c-faeb-46e5-afe9-227289738dc4___RS_NLB 3749",
  "bfa4998b-ab2a-439a-8e65-3501db446811___RS_NLB 3704",
  "bfd70e3b-3556-4dab-b5c2-64a6b0dfa9b5___RS_NLB 4110",
  "bfe59dc5-c1fd-4fb0-8c6f-ed9b1d9da11b___RS_NLB 0786",
  "c00aaa9f-85ca-4cb0-9734-b55296153840___RS_NLB 4081",
  "c0b461c8-cb13-47c8-8e35-25996de22e7e___RS_NLB 3618",
  "c0c7e666-4b2b-46f1-b315-274cfee65342___RS_NLB 4014",
  "c0d0dfd0-2167-4cde-b90e-170703d962f0___RS_NLB 4255",
  "c141f6bd-5696-4388-abd0-cd413e43dfe1___RS_NLB 4208",
  "c2e5ab9e-e705-4f13-9c50-1243e8ea0255___RS_NLB 3555",
  "c35eae34-63ef-4f1d-acc6-7dd2aa23deb6___RS_NLB 0825",
  "c3b2b118-3ee2-46cf-8c21-4d93523f4ccb___RS_NLB 3671",
  "c3c358d5-4ca3-4dcb-9725-4d6f022c41e1___RS_NLB 3522",
  "c46d21be-77a3-441e-b301-de0da5b99898___RS_NLB 4234",
  "c4eb96b2-1345-46d0-a796-161042da21df___RS_NLB 0824 copy",
  "c72b436a-2b40-4e4c-8a87-275ec5c90bea___RS_NLB 0827",
  "c8132bc8-730e-4920-a9f2-a0d05985d3dd___RS_NLB 0801",
  "c8b11ed7-bf90-4eec-aa68-45c577c30896___RS_NLB 4046 copy 2",
  "c94c8b78-b8e1-45cf-bac8-80d8a0c6b64d___RS_NLB 0781",
  "c9c8abd4-f8e9-45b7-9660-afeccb515539___RS_NLB 4235 copy",
  "ca3919bb-4e04-4fb2-a2d9-22e798d6ca60___RS_NLB 3970 copy",
  "ca52142c-e23d-4445-881b-dd95e94cb208___RS_NLB 3652",
  "caee29c3-dc6d-4fed-a4cf-3d025990c3ef___RS_NLB 4159",
  "cb5c61df-e4de-4667-88d2-85b43d708b59___RS_NLB 4075",
  "cc3c5678-ed5a-43f6-bf74-5c13e9ee71ee___RS_NLB 3651",
  "cc76de91-e4bf-4d71-a74f-0f2a28ce7f9b___RS_NLB 4013",
  "cc926ea7-7824-4f6d-8c2b-837771ffdbee___RS_NLB 4142",
  "cd0b6059-1250-4230-bdd6-488f75143df6___RS_NLB 3596",
  "ce1af083-ea99-4aeb-9f66-c3bd1ecd6be3___RS_NLB 3485",
  "ce1bbcb3-e17b-4b81-bf54-9c5d14d3faee___RS_NLB 4248",
  "cea4e177-947e-466f-9b0f-fc2adae6a02e___RS_NLB 4198",
  "cebcbe6b-0ed3-4397-b94a-bd40ab1094e7___RS_NLB 3698",
  "cf3d2eeb-aa81-4956-bd18-fa7e10e8ba25___RS_NLB 4085",
  "cfa52648-e9ac-45da-8046-e96046ff948a___RS_NLB 0804 copy",
  "cfec0626-7a50-4411-a6b9-f63a49e4d2eb___RS_NLB 3746",
  "d048f7c9-4398-4df3-a437-6ee08b579e87___RS_NLB  4507 copy 2",
  "d08114ac-a98f-41eb-82f9-fdfa91446ed7___RS_NLB 4062",
  "d0ec3b78-8ea4-4938-bf6d-1baf5f0d640f___RS_NLB 4250",
  "d1265756-90d5-4126-943b-4bff5b1e8277___RS_NLB 3904",
  "d1718bf2-fb43-42c8-a143-a78ea89017a7___RS_NLB 3491",
  "d1dc8657-22f9-42f8-a467-0dc1f0e2ed2e___RS_NLB 3643",
  "d1fbc905-f7df-4f45-b228-0a94b00f16ee___RS_NLB 3488",
  "d22ff633-ea78-4d2f-ace7-cea0035e48d9___RS_NLB 0778",
  "d281dba8-6b57-4bec-9e86-4b5638f4d4dc___RS_NLB 3860",
  "d317e749-6b6b-44b2-9db8-6ad489cf8d2e___RS_NLB 3750",
  "d330ac59-dc5f-49d4-9529-a4ebbc34d46b___RS_NLB 3531",
  "d334d37f-a75c-47d9-ad48-3d37c32ea3c9___RS_NLB 3582",
  "d36dfeef-82b0-4bfc-b202-0abe50242a21___RS_NLB 3928",
  "d3cac9cc-2933-4c51-a3cf-005b717ad180___RS_NLB 0830",
  "d3d7a5a3-ee3f-4654-8e90-8e722d433972___RS_NLB 3553",
  "d413b5ce-75a2-4e7a-977a-72318cdbf476___RS_NLB 3668",
  "d469389c-709c-485f-867b-a0243aaa1c88___RS_NLB 3756",
  "d4e0ed9b-03b3-487a-9fc5-1a1be771367f___RS_NLB 4256",
  "d51c7157-2d00-4535-b22f-a82238b8b1cc___RS_NLB 0759 copy",
  "d5e92223-9fac-4b99-9021-f8867c378fba___RS_NLB 3563",
  "d5ecb0c7-cdd5-46f4-bced-94f35423a245___RS_NLB 4194",
  "d612943e-54f2-435d-ade7-973c02e234e9___RS_NLB 4026",
  "d62e34ed-d8ff-4678-9534-fbcadfb91703___RS_NLB 3726",
  "d66d0212-e2bc-48d4-a7ba-8f0bc6549280___RS_NLB 3621",
  "d675c185-9018-4693-9bb3-d5f325cd0c2c___RS_NLB 3551",
  "d6fa356e-05e8-45fb-9084-54f7cc7db81e___RS_NLB 0822 copy 2",
  "d77fa827-9282-439b-8e87-5759c64cc852___RS_NLB 4285",
  "d7937bfb-4767-4cb4-a2f4-85cd144240fe___RS_NLB 4690",
  "d7fdbb58-9e9b-4d2e-8b50-9da0e6a8a8cf___RS_NLB 0807 copy 2",
  "d808adb1-8f84-4b4c-84ca-d3f93a349650___RS_NLB 3534",
  "d82213a9-eff4-4d5c-a2a4-c55cc8a034e3___RS_NLB 3748",
  "d8a2c04c-5d2b-485c-aaff-1440c0cd65e1___RS_NLB 3695",
  "d90d1012-715f-4cf5-8e47-7fe6b73ce859___RS_NLB 3715",
  "d95f6c9c-1c62-47d6-a643-093cd3402162___RS_NLB 3942 copy 2",
  "d9b926c7-d24d-4773-b1c6-c6b911984121___RS_NLB 3600",
  "da061484-a583-40ad-bbbd-cab07dd81d80___RS_NLB 3536",
  "da24c7cf-a119-46e9-b34a-049d21fe610c___RS_NLB 0831",
  "dac28004-9243-45ab-a3ff-03edff04e8d1___RS_NLB 3613",
  "dafabebd-c6d1-4f69-a7a9-45f5c959f9a1___RS_NLB 3571",
  "db19f539-05e8-4cd8-a9bc-57fb97ce0f41___RS_NLB  4443 copy",
  "db43f70c-19bc-4913-b8ae-90c7dca09411___RS_NLB 4035",
  "dbc0dc71-c477-4d79-9c13-479bbbe783c8___RS_NLB 3931 copy",
  "dc425939-95f6-4cec-b669-73d61947c66b___RS_NLB 0834",
  "dc5013d6-de04-4079-93f1-320d6b765cd3___RS_NLB 0822 copy",
  "dd430eb3-4f5c-45a9-9f18-c5ee82073ec4___RS_NLB 4195",
  "dda73df7-96ed-453c-8d53-21549a980bc7___RS_NLB 4182",
  "ddd3bb1b-ef88-4426-a7bb-2b61422df6e2___RS_NLB 4057 copy3",
  "def2064a-8470-447e-bfb2-f76ecb3006cd___RS_NLB 0789",
  "e0246bcb-949d-4ea8-b517-f6dfb7e7526d___RS_NLB 4107",
  "e04b792b-62d1-4447-b9e2-cf3b8f565883___RS_NLB 3879",
  "e09f88c6-ca04-4f63-bc86-e18a6cbad85d___RS_NLB 3574 copy 2",
  "e11c9b3e-d9bc-48b7-8d05-c1d99ba2da52___RS_NLB 3851",
  "e15e053c-4b0a-48a3-a622-c2395bd93b64___RS_NLB 4008",
  "e20d6d7c-2508-4a75-9143-5c49d0c02d12___RS_NLB 3604 copy",
  "e22976d4-ff48-4672-ad70-e17d50b65b1c___RS_NLB 3688",
  "e2d99dbe-c4ff-46d8-8e09-38a19f10d2b5___RS_NLB 3533 copy",
  "e2faa2e3-433a-4ee3-9b64-8de4ad2947e1___RS_NLB 3795",
  "e2fc31a8-6b0e-4c15-9fba-2024d2295827___RS_NLB 4229",
  "e379b939-5325-400c-97ea-393159297fc0___RS_NLB 4184",
  "e4a1bfbf-22a0-4455-98e2-a1bd0a243a94___RS_NLB 3975",
  "e59d2407-e225-47d5-96a7-6cdcc3868499___RS_NLB 4100",
  "e5e4cd23-60b4-4b94-bd47-31495d8e7263___RS_NLB 3514 copy",
  "e60c841c-2a10-4a27-a2e9-2604fb9ead83___RS_NLB 4192",
  "e63578a0-74d4-4634-b377-aaaf3c135324___RS_NLB 4200",
  "e6f10c6e-9cd7-4ad1-98d8-0fb8038e4c1a___RS_NLB 0820",
  "e7510f80-69fa-49d3-b6c5-037a43c0622a___RS_NLB 3570",
  "e7f21b10-34e7-408e-a434-2b5d9e5e4eeb___RS_NLB 0761",
  "e8527ec9-d83a-49c6-b0ed-1833e6350e23___RS_NLB 3796",
  "e863323a-5d81-42c8-8503-dbdb00452e59___RS_NLB 3554",
  "e8fe9001-df63-40de-bc41-fc250863a705___RS_NLB 4205",
  "e9b6bee3-c097-4823-b057-c7ed167f072c___RS_NLB 4151",
  "e9c01f8e-6d45-4243-b274-6036ab3e5529___RS_NLB 0774",
  "e9c3b693-1824-41bb-803c-2730cd06f9e5___RS_NLB 4152",
  "ebb837be-867e-417a-8a0a-04fd3f0570ba___RS_NLB 4087",
  "ebcfe20f-f00e-49ad-a470-4b4cc40b815c___RS_NLB 3742",
  "ecaf0f9a-338f-4065-bc84-7185d8394552___RS_NLB 3898",
  "ed700cb4-fa84-4c56-bcf4-f2c21ce53e12___RS_NLB 4040",
  "ed87978b-71f0-4ab8-b1d3-e7038c0dfa16___RS_NLB 3494",
  "ed966f7f-71e9-4dc2-9689-32654a609956___RS_NLB 4046 copy",
  "ee389c10-aaf2-42d5-bd7e-aabfb9acd42b___RS_NLB 4183",
  "ee4269d5-9492-4863-a283-17396a086e0d___RS_NLB 3672 copy",
  "eebc39f9-9bb0-4509-91ae-135a4f0563f2___RS_NLB 3677",
  "ef2883c1-77cd-4b12-98bc-db371e596eb7___RS_NLB 3989",
  "ef51615e-208a-4af4-a392-29abe4c5d008___RS_NLB 4062 copy 2",
  "efb9c96a-22d4-4217-bf2b-7c0f5d97d277___RS_NLB 4057 copy 2",
  "f06ca1ec-25c8-4d0f-ae7c-6bb0ffb7e0b0___RS_NLB 3584",
  "f09c5304-da39-4f77-a7d5-8c101b286509___RS_NLB 4019",
  "f13d43cb-e365-4df6-aac8-746e4d6d7dab___RS_NLB 3528",
  "f23ac0bb-a341-45d3-9e3d-8180658ddc96___RS_NLB 4230",
  "f283c7ad-19cf-4e5f-a2cb-d1123610b077___RS_NLB 3573",
  "f301b26e-ec9c-4d03-94fa-28a2a8fe617a___RS_NLB 4254",
  "f3141371-5c55-4cf4-988a-6c322de0c6ec___RS_NLB 3486 copy 2",
  "f339c8f8-0de3-427f-86ce-2d3258794ad2___RS_NLB 4176",
  "f38dbeb3-7140-4470-911f-7545b495776f___RS_NLB 3473",
  "f3a574c8-043a-4804-bdba-addb50450720___RS_NLB 3553 copy",
  "f4a5c19b-23ae-44f2-8e19-badb0f573f1b___RS_NLB 3535",
  "f52ec293-3749-4e11-bcab-2935dcd6eb6d___RS_NLB 4150",
  "f546ba30-4b0b-4906-a527-bb4834df87cd___RS_NLB 3558",
  "f553d7cf-9c9f-46bf-b99a-8e3505afd96d___RS_NLB 3873",
  "f5773153-728e-4631-8cc8-1bc06412df89___RS_NLB 4056",
  "f6b47d16-cfbc-4786-bc6d-78a2bc037a43___RS_NLB 3545",
  "f7cf0608-2278-4a38-a37e-1356bd359651___RS_NLB 3490",
  "f7f43a8d-2f8f-49c1-ad47-c01d9396bd78___RS_NLB 4117",
  "f81a1119-4ccb-46b6-aff7-29998828be59___RS_NLB 4187",
  "f89ead10-10be-4102-82f8-35810eb4a3d1___RS_NLB 3557",
  "f9276b30-6706-47b2-9396-3799808cc29a___RS_NLB 4262",
  "fa30c8e9-b74d-49a2-8611-c9b9098e21c3___RS_NLB 3507",
  "fae01c0a-db82-4189-9193-bdad09ede531___RS_NLB 4165 copy",
  "fb1e8075-1546-4c0a-bef1-e18642415358___RS_NLB 3966",
  "fb83c22d-b379-48c7-bccc-aebdfd19eda0___RS_NLB 4252",
  "fc19258c-634e-4cc0-b6cf-0144c550ed80___RS_NLB 3834 copy",
  "fc84ca09-d6c7-493e-957b-c46b2055b702___RS_NLB 3836 copy 2",
  "fcdc4514-7dfa-4003-aebd-27692e0853dd___RS_NLB 3705",
  "fd4eff3c-6bfb-42ad-bc34-5ae0f7e30c0f___RS_NLB 0769",
  "fdca88b7-bd2e-472a-b48f-7a5347d72f04___RS_NLB 3511",
  "fdff294e-df6e-42eb-9bcf-edecf4002891___RS_NLB 4158",
  "ff1721eb-609e-4476-983d-a372b7469ab0___RS_NLB 4039 copy 2",
  "ff8577c3-ad10-4272-84f6-b7c6d6042d1f___RS_NLB 3506",
  "ffbc3745-0a6d-4145-8f92-781edc626d9f___RS_NLB 4260"
 ],
 "Orange___Haunglongbing_(Citrus_greening)": [
  "1a3a5d93-5025-473f-9ac2-5f963e50c5f9___UF.Citrus_HLB_Lab 0692"
 ],
 "Peach___Bacterial_spot": [
  "29eb5a26-6502-470d-b94d-339f7fa627f4___Rut._Bact.S 3339",
  "2e53ece1-fd49-45cc-9d03-d9e086e49264___Rut._Bact.S 3377",
  "3a26119f-a70d-484e-b769-50fb1174f898___Rut._Bact.S 3357",
  "4d3da774-29ff-479a-a6a3-fff095012dff___Rut._Bact.S 3492",
  "afcce0ff-9842-4cb9-839e-3387e4f292cb___Rut._Bact.S 3403",
  "bf589182-168b-4d98-8f6c-fb082320fc61___Rut._Bact.S 3340",
  "d2ee77df-e89d-4110-9873-36e15d8850e5___Rut._Bact.S 3343",
  "dcb4937e-813b-4c7b-b71b-2e487870504d___Rut._Bact.S 3369",
  "f90d88b9-81ff-445c-b2d4-0e1dcc366400___Rut._Bact.S 3373"
 ],
 "Soybean___healthy": [
  "326965ac-9077-4a45-b40e-4824bd8fec1f___RS_HL 5697",
  "42082722-38c1-4ee6-b60c-8259452b0c45___RS_HL 6949",
  "902b58e9-d74f-4ffd-ba17-30e91ab4819f___RS_HL 6237",
  "b22f2758-59c2-48dc-8933-da544082304c___RS_HL 6283"
 ],
 "Squash___Powdery_mildew": [
  "00cfd4c7-4ccd-45a8-813c-f1cb01147aed___UMD_Powd.M 0108",
  "0582102f-25c7-43e6-af80-3af172691276___UMD_Powd.M 9750",
  "0b907a4e-2d99-4775-ae65-4c8c5c53ba9a___UMD_Powd.M 0074",
  "0e8556f8-bf5e-406b-ad73-7c2b0c239959___UMD_Powd.M 9746",
  "0ef07958-0cdf-4dad-a4de-b4d5a773f223___UMD_Powd.M 9978",
  "109fbbd3-2020-4166-8258-2034978d3ca6___UMD_Powd.M 9837",
  "12aa63f4-ae45-48a8-bedc-54e1e7c78144___UMD_Powd.M 0059",
  "13d1aacb-4f5d-43b1-aca8-9307b4cb1754___UMD_Powd.M 9789",
  "19ba7276-1f19-422e-90ae-5c45a0d0e965___UMD_Powd.M 9777",
  "1a79bba6-f59b-439f-9b4d-855345db220a___MD_Powd.M 0028",
  "1c742e44-0307-4e0a-b7eb-90d2ceeadec5___MD_Powd.M 0121",
  "1d87ede4-0884-4ce0-8ad8-101769a42934___UMD_Powd.M 0225",
  "1dc6e12f-2a57-4177-b450-9488bb2de23c___UMD_Powd.M 9979",
  "1fc38e9d-c232-4191-a9b8-2c4b3011be23___MD_Powd.M 0012",
  "2203a2ef-1bbd-44d6-afb4-a903ffe68439___UMD_Powd.M 0115",
  "27a8a678-6a6b-4d26-9f3d-d410aa806256___UMD_Powd.M 9888 copy",
  "2a6d3bdb-b8de-454f-ad74-49861f6f8499___UMD_Powd.M 9773",
  "2ca04622-a708-4e0b-a3d9-ee0f685708b5___MD_Powd.M 0052",
  "2dca0657-3a77-4e8e-89ca-9f70650bc80f___UMD_Powd.M 9886",
  "332fde6a-5e8b-4a8f-85a0-e3550196774d___MD_Powd.M 0180",
  "3887c3f3-92ed-46bc-acf5-ee41c905575b___UMD_Powd.M 0399",
  "3da1cc34-9229-4b8b-82d0-d764c98c9213___UMD_Powd.M 0088",
  "3f1f3964-2506-4bbe-be90-8c7edf6eb607___UMD_Powd.M 9780",
  "412c48c1-fc0d-432c-a9fe-228124a882f8___UMD_Powd.M 0328",
  "47b5013f-d5ad-4bb2-bc47-c4a8e6fac4d6___UMD_Powd.M 0016",
  "4b121e08-c44d-427e-ae19-a072234da630___UMD_Powd.M 9829",
  "51fc414c-c6ec-4370-835f-f02fead6813e___UMD_Powd.M 0022",
  "5b83d919-e94e-4788-ac46-6ff1d617d476___UMD_Powd.M 9974",
  "6144bf6a-4ffd-4cb7-8dea-8ccb73024937___UMD_Powd.M 9905",
  "6338484a-a0da-441e-a79c-a088de6b2615___UMD_Powd.M 0500",
  "636f0770-e5f6-4e6f-a46f-5d984e4adbca___MD_Powd.M 0122 copy",
  "6597567f-5d09-4643-84cc-dea248b75551___UMD_Powd.M 0040",
  "65ed1fe2-c507-4590-8767-2a557dd05195___UMD_Powd.M 9778",
  "676f8bc9-46f3-40ea-92c4-7595670efa23___UMD_Powd.M 9758",
  "6ba600b8-626a-4cd4-870e-ad9a47aa464c___MD_Powd.M 9903",
  "7671fde9-6de2-426b-bb00-edc027a6bc32___UMD_Powd.M 9772",
  "76ab5756-f5f6-4d93-8f09-8f1ee910cc55___UMD_Powd.M 0066",
  "7a6cdd29-9f54-44eb-b7ec-c2515573fdc9___UMD_Powd.M 9792",
  "7cb48f2b-cb6c-4266-aec2-0ac889b7beb5___UMD_Powd.M 0146",
  "7d0432e7-133f-4553-b1d7-7310c682a8c6___UMD_Powd.M 0029",
  "7d6b0f63-f46a-4a6e-a6d5-d4724baa23ca___UMD_Powd.M 0065",
  "7eac583e-2703-4c66-823b-5725fc4cbe2a___MD_Powd.M 0025",
  "7ef62ec7-069c-4d1a-9f96-e6840cf39eb4___UMD_Powd.M 0039",
  "7f404132-e36b-45c5-9aff-4136cccaf5c6___UMD_Powd.M 9832",
  "8444d498-ebc5-41c9-9f1b-5d6730ce6efe___UMD_Powd.M 0010",
  "8a145ed3-c31c-47f9-95c2-9bd399ac3e1f___UMD_Powd.M 0042",
  "91b54451-c1b4-43fe-a975-c3e23ba6ffde___MD_Powd.M 0179",
  "92c5dd28-ea56-4d45-a61b-1d95dbe84425___UMD_Powd.M 9835",
  "93572ef2-28ef-4bcc-9a9e-fa5f508e89b1___UMD_Powd.M 0063",
  "9363ff26-0b13-4a6a-8568-07aa13310dfc___MD_Powd.M 0172",
  "964cf17d-da16-4719-a7f4-aeefa86dd84b___MD_Powd.M 0122",
  "971fc20d-d566-4ae9-ac26-91c41e238846___UMD_Powd.M 9739",
  "9750b896-780b-4c18-a484-6b6acb720425___UMD_Powd.M 0234",
  "982477f4-224f-44a6-92bb-aa31dcdb3048___UMD_Powd.M 0475",
  "9b00dd6f-2174-410b-9cd0-ab53ea0e6025___UMD_Powd.M 0015",
  "9b338af8-ad6b-4889-a4f7-b6f973c647f7___UMD_Powd.M 0101",
  "a05230cc-91ed-4681-b670-122d1ced44b9___MD_Powd.M 0048",
  "a4d20de2-7691-4986-a452-a7ccf2a69663___UMD_Powd.M 0224",
  "a63b7b04-5565-4f6e-bb60-20252206a581___UMD_Powd.M 0285",
  "a881d099-5b4b-4b0d-8d13-8a6f1e84ddb6___UMD_Powd.M 0590",
  "ad323f3d-6d89-4c90-aee5-97f33a73f825___UMD_Powd.M 9888",
  "af272ab6-656a-43c4-8607-2b89001b5f72___UMD_Powd.M 0081",
  "b2521c2e-d2d2-4c8f-8f28-0372dec3e940___UMD_Powd.M 0148",
  "b55e1888-c147-4397-a4ed-43afa9a3855e___UMD_Powd.M 9823",
  "b6d65439-87ca-4876-85c8-96473c18c935___UMD_Powd.M 9757",
  "be1b72ef-e169-4ffb-b872-05e89ef420a7___UMD_Powd.M 0277",
  "bf430bca-bb53-447e-a1ae-dac307a3268b___UMD_Powd.M 0268",
  "bf4e3214-a008-45dd-bc72-178b51eadf14___UMD_Powd.M 0230",
  "c0034f5c-77ae-4876-a74e-976877c78ddc___UMD_Powd.M 0023",
  "c0589deb-14f5-4649-9e12-b93be348e757___UMD_Powd.M 0591",
  "c2514221-fbf1-4610-9af2-0dc81d306446___UMD_Powd.M 0256",
  "c4461a00-67aa-445f-9462-e65c660dc2b9___UMD_Powd.M 9953",
  "c48a1038-78d2-4c37-ba13-54903798037f___UMD_Powd.M 0037",
  "c5455cd1-9c8a-4051-aca1-ed1f1f784bea___UMD_Powd.M 0226",
  "c556ad9e-d84f-4d18-a4c8-681acb283999___UMD_Powd.M 0038",
  "cbda95e0-1a50-44c2-8f4b-3d3adef9e0c2___UMD_Powd.M 9774",
  "cce0c23e-1071-4ecb-8383-509eeccc5f03___UMD_Powd.M 0406",
  "d02f9c93-bcec-4c98-bbe1-d4c329c65fb2___UMD_Powd.M 0183",
  "d236d90e-46f2-4a28-aaf4-d43b178e91a2___UMD_Powd.M 9944",
  "d38d0144-2ca4-43fc-ad2c-bedec1a22a63___UMD_Powd.M 9798",
  "d39238b0-d47d-46a8-874a-13d37d01689a___UMD_Powd.M 0400",
  "d42b8779-0f87-4285-b968-d53e7f2f1287___UMD_Powd.M 0128",
  "d6ed345c-6ae8-45c4-b26c-69410c6a00d0___MD_Powd.M 0169",
  "dd88f921-810e-4578-9929-d70d3d306bd7___UMD_Powd.M 0275",
  "de8c8a94-2575-4f87-b736-556b04667719___UMD_Powd.M 0041",
  "debac46c-898f-4a11-98b0-e3ee9ba71c20___UMD_Powd.M 9755",
  "df3ee91a-2d45-4dcf-a410-18ae6ab7aacc___MD_Powd.M 0118",
  "dfe23030-072f-475f-9fb1-3cc82e92a14c___MD_Powd.M 0171",
  "e07472dd-35cc-4339-9045-1ab686751d52___UMD_Powd.M 0246",
  "e26c8d14-d466-4c4f-abb3-ab4ea04c134d___UMD_Powd.M 0618",
  "e6988ba5-e3b5-4bee-8fe2-9a078e701a2e___UMD_Powd.M 9946",
  "e76e069c-8e9c-4d2e-a2cf-5b84683f8c1e___MD_Powd.M 0124",
  "e8d430af-d48f-49fe-bf73-4372f29ea93b___UMD_Powd.M 0080",
  "eb89913c-0e80-40cd-9d87-f2e3c99ce7ae___UMD_Powd.M 0558",
  "ec1d9d9d-9e8a-4fcc-9084-4903b7921044___UMD_Powd.M 0569",
  "eefb164e-a75d-453d-90d4-b5387c3f1003___UMD_Powd.M 0028",
  "efcbe3b9-651d-44bc-8527-ddfe61543ccb___UMD_Powd.M 0014",
  "f0e34845-44b7-4a1a-85cd-1a2b90438997___UMD_Powd.M 0552",
  "f5015f94-4ed5-4f0c-8111-51a32e8a09d6___UMD_Powd.M 9838",
  "fb4509bc-5412-444c-be2c-dcdaf9ad6e96___UMD_Powd.M 9711",
  "fbb3d787-da7f-41a4-b0c4-8863c97929a7___MD_Powd.M 0116",
  "fd4c5cc2-624b-4a62-a33f-79f7013f9856___UMD_Powd.M 0403"
 ],
 "Tomato___Late_blight": [
  "06412832-90d3-4713-9e82-f2cca52dadc8___GHLB2 Leaf 8734",
  "08a723e4-6b51-4395-8013-b1f3f03709b8___GHLB2 Leaf 8967",
  "0ed7d5f6-9394-4ae7-87db-3de58f469263___GHLB2 Leaf 8732",
  "14113767-0c24-4607-97bc-04dccb35a23c___GHLB2 Leaf 8966",
  "146b33cf-e412-48fd-9366-ad726b51889e___GHLB2 Leaf 157.1",
  "1aa88c5b-384f-43cf-8046-0d32d26a1133___GHLB Leaf 24 Day 12",
  "36193f11-7789-416d-b0dd-0cae49f74836___GHLB Leaf 23.2 Day 18",
  "531d1dc7-d864-4464-b2ad-1fc76f825021___GHLB2 Leaf 8742",
  "56c7bdb1-9b7e-40fc-8ec5-97e20770426b___GHLB2 Leaf 8951",
  "7b343a4c-5fbc-40f6-83da-ccae49ebd5fb___GHLB_PS Leaf 35 Day 13",
  "7edf8e92-d212-46f4-843e-0c08bd263f73___GHLB2 Leaf 8944",
  "7f790574-ac82-4a3e-9a4d-02cd7762dce2___GHLB2 Leaf 8948",
  "8130621c-8ff8-4fd9-bd54-765491adc31f___GHLB2 Leaf 8822",
  "81b4d72b-4124-45d4-b96a-0ec13ca7f376___GHLB2 Leaf 8989",
  "8275c408-1ce2-4b4b-a6de-b7a0b373ad05___GHLB2 Leaf 8741",
  "830fab73-df27-4430-ba0a-2b638be3b6b4___GHLB2 Leaf 8946",
  "920f60f5-6435-427f-a855-5e66552dd8b9___GHLB2 Leaf 9020",
  "94b06669-5f80-4a76-9eae-8d34f6489f15___GHLB2 Leaf 8839",
  "9cb4cbdd-147f-4cfe-a30a-dd003520f0dc___GHLB2 Leaf 8988",
  "a202fc9d-50ed-43eb-9a75-348348549924___GHLB2 Leaf 8736",
  "b19055d9-6a5d-4b73-90a5-3b3685ac344a___GHLB2 Leaf 8729",
  "b657dc7a-df76-40e1-bc87-2d61b81040a8___GHLB Leaf 27.1 Day 12",
  "c933f19c-7b00-4dca-b2c4-94d3b9212f92___GHLB2 Leaf 8985",
  "eb5ce385-86d7-439c-b3e4-ca375a739afa___GHLB2 Leaf 8603",
  "ecfdf723-18dd-46d5-9d22-a54289e9d765___GHLB_PS Leaf 31 Day 12",
  "ed974f5b-e999-46a9-9e22-da51da2766b6___GHLB_PS Leaf 8 Day 16",
  "f15b868b-23ae-4e87-8053-6199c363e079___GHLB2 Leaf 8984",
  "fcd37e39-9a92-4896-8673-af928a380078___GHLB2 Leaf 8733",
  "fd26b445-5839-4337-858c-a91dfdd812cc___GHLB2 Leaf 8743"
 ],
 "Tomato___Septoria_leaf_spot": [
  "2cb00d76-a875-4c94-a383-6e55534d1506___JR_Sept.L.S 8507",
  "3d638a43-43eb-4f93-85bf-b1cc41ab117f___Matt.S_CG 0751",
  "52b3119d-4f5e-41b0-948f-7a1d3b1341eb___JR_Sept.L.S 8495",
  "604db609-ce1b-48e0-bf73-4f9c53ffa051___JR_Sept.L.S 8418",
  "650abe19-6687-4d96-ad08-1c887b8c2a62___JR_Sept.L.S 8497",
  "79f0e1e3-7275-46a5-b4de-fbf5410dae80___JR_Sept.L.S 8482",
  "8c73d17b-4821-4e52-9627-3914f7c89bce___JR_Sept.L.S 8481",
  "e94aaf1a-0c1b-4020-87f0-93c74f929f1c___JR_Sept.L.S 8496",
  "f2be0eb5-213e-44f1-b618-4d329e04b6a4___JR_Sept.L.S 8494"
 ],
 "Tomato___Target_Spot": [
  "e475a5f4-8166-4b18-9f29-8a7ccb541bbb___Com.G_TgS_FL 9878"
 ],
 "Tomato___Tomato_Yellow_Leaf_Curl_Virus": [
  "3963a2e9-e6cc-4c8c-b8d2-96c7b2aa2f5a___UF.GRC_YLCV_Lab 01587",
  "97c12f01-9f88-4895-b0da-90c882f3f79a___UF.GRC_YLCV_Lab 01588"
 ]
}
print(RAW_DIR, sum(map(len, IMAGES.values())), "images")

In [ ]:
import io, zipfile

import cv2
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForImageSegmentation.from_pretrained(MODEL_ID, trust_remote_code=True).to(device, dtype).eval()
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
to_pil = transforms.ToPILImage()


def board_colour(rgb):
    # Median of low-chroma, mid-brightness pixels, i.e. PlantVillage's grey board.
    mx, mn = rgb.max(-1).astype(int), rgb.min(-1).astype(int)
    board = (mx - mn < 18) & (mx > 70) & (mx < 235)
    return np.median(rgb[board], 0) if board.mean() > 0.02 else np.array(DEFAULT_BOARD, float)


def pad(rgb, ratio, rng):
    h, w = rgb.shape[:2]
    H, W = round(h * ratio), round(w * ratio)
    canvas = board_colour(rgb) + rng.normal(0, 3, (H, W, 3))
    top, left = (H - h) // 2, (W - w) // 2
    canvas[top:top + h, left:left + w] = rgb
    return Image.fromarray(np.clip(canvas, 0, 255).astype(np.uint8)), (left, top, left + w, top + h)


def colour_mask(rgb):
    # Rough leaf mask from Lab b* - a* (leaf is green-to-yellow, the board neutral grey). Too blocky to use as a
    # cutout, but good enough to tell a plausible BiRefNet mask from an empty, lesion-only or whole-frame one.
    lab = cv2.cvtColor(cv2.GaussianBlur(rgb, (5, 5), 0), cv2.COLOR_RGB2LAB).astype(np.int16)
    yg = np.clip(lab[..., 2] - lab[..., 1] + 128, 0, 255).astype(np.uint8)
    otsu, _ = cv2.threshold(yg, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fg = (yg > 128 + np.clip(otsu - 128, 3, 15)).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)
    if n <= 1:
        return fg.astype(bool)
    areas = stats[1:, cv2.CC_STAT_AREA]
    fg = np.isin(labels, 1 + np.flatnonzero(areas >= 0.25 * areas.max())).astype(np.uint8)
    # Fill enclosed holes unless smooth (smooth = board showing through a real hole; textured = lesion or mildew).
    n, labels, stats, _ = cv2.connectedComponentsWithStats(1 - fg, connectivity=4)
    h, w = fg.shape
    x, y, bw, bh = (stats[:, k] for k in (cv2.CC_STAT_LEFT, cv2.CC_STAT_TOP, cv2.CC_STAT_WIDTH, cv2.CC_STAT_HEIGHT))
    enclosed = (x > 0) & (y > 0) & (x + bw < w) & (y + bh < h)
    enclosed[0] = False
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    texture = np.abs(gray - cv2.GaussianBlur(gray, (0, 0), 2))
    for i in np.flatnonzero(enclosed):
        hole = labels == i
        interior = cv2.erode(hole.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
        if not interior.any() or texture[interior].mean() > 5:
            fg[hole] = 1
    return fg.astype(bool)


def candidates(paths):
    # Yields (path, image, {ratio: soft mask at the image's own size}).
    rng = np.random.default_rng(0)
    for i in range(0, len(paths), BATCH_SIZE):
        batch = [(p, Image.open(p).convert("RGB")) for p in paths[i:i + BATCH_SIZE]]
        found = [{} for _ in batch]
        for ratio in RATIOS:
            padded = [pad(np.asarray(im), ratio, rng) for _, im in batch]
            x = torch.stack([transform(im) for im, _ in padded]).to(device, dtype)
            with torch.no_grad():
                preds = model(x)[-1].sigmoid().float().cpu()
            for f, (canvas, box), pred in zip(found, padded, preds):
                f[ratio] = to_pil(pred.squeeze()).resize(canvas.size, Image.BILINEAR).crop(box)
        for (p, im), f in zip(batch, found):
            yield p, im, f


paths = [next(f for f in (RAW_DIR / c).iterdir() if f.stem == s) for c, stems in IMAGES.items() for s in stems]
manifest = {"ratios": RATIOS, "min_iou": MIN_IOU, "chosen": {}, "fallback_full_image": []}
with zipfile.ZipFile(WORK_DIR / "segmented_padded.zip", "w", zipfile.ZIP_STORED) as zf:  # PNGs are already compressed
    for n, (p, image, found) in enumerate(candidates(paths), 1):
        key = f"{p.parent.name}/{p.stem}.png"
        ref = colour_mask(np.asarray(image))
        ious = {r: float((m & ref).sum() / max((m | ref).sum(), 1)) for r, m in ((r, np.asarray(m) > 127) for r, m in found.items())}
        best = max(ious, key=ious.get)
        if ious[best] >= MIN_IOU:
            mask = found[best]
            manifest["chosen"][key] = {"ratio": best, "iou": round(ious[best], 3)}
        else:
            mask = Image.new("L", image.size, 255)
            manifest["fallback_full_image"].append(key)
        image.putalpha(mask)
        # Zero the colour under fully transparent pixels so the background can't resurface via .convert("RGB").
        rgba = np.array(image)
        rgba[rgba[..., 3] == 0, :3] = 0
        buf = io.BytesIO()
        Image.fromarray(rgba).save(buf, format="PNG", optimize=True)
        zf.writestr(f"segmented/{key}", buf.getvalue())
        if n % 100 == 0:
            print(n, "/", len(paths), flush=True)
(WORK_DIR / "manifest_padded.json").write_text(json.dumps(manifest, indent=1))

from collections import Counter
print(len(paths), "images;", len(manifest["chosen"]), "segmented,", len(manifest["fallback_full_image"]), "kept whole")
print("kept whole by class:", Counter(k.split("/")[0] for k in manifest["fallback_full_image"]))
print("chosen ratios:", Counter(v["ratio"] for v in manifest["chosen"].values()))